<hr>

# ℹ️ DATA EXPLORATION


<style>
h1 {
    text-align: left;
    color: blue;
    font-weight: bold;
}

</style>
<hr>

```text
Understand data and take Notes for Data cleaning, Analysis, ML:
    - Valeurs Foncieres (2020-S2 to 2025-S1) 6 large files in the format of .txt
    - arrondissements.csv (paris)
    - communes.csv (outside of paris)
    - Amentities Tool https://overpass-turbo.eu/
    - Rental data in/outside Paris APIs https://www.observatoire-des-loyers.fr/
```

---
<style>
h3 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

### 📂 IMPORTs

In [1]:
# import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import seaborn as sns

plt.style.use('ggplot')
pd.set_option('display.max_columns', 200) # to display all columns in the dataframe

---
## **💰 CSV VALEURS FONCIERES 2020-2025**

PROPERTY SALE & PRICE HISTORY

What it contains:
- Transaction price (€) 
- Property size (m²)
 - Number of rooms 
- Exact address (geocodable) 
- Property type (apartment, house, etc.) 
- Sale date 
- Historical price evolution (5–6 years)

DVF dataset is used for:

- 📈 Price prediction
- 📊 Market trends
- 🤖 ML models

🎯 TARGET VARIABLE = Valeur fonciere (price)

to clearly see:
- 📈 Market growth over 5–6 years
- 🏠 Difference between different types of properties: houses vs apartments vs ... 
- 📉 Market dips (e.g., COVID effects, gaz inflation cuz of Iran/Israel & USA war)

---
La France en quelques chiffres clés:
- Superficie : environ 543 000 km² pour la métropole, 120 000 km² pour les DROM et 4 600 km² pour les COM (667 000 km² au total).
- Nombre d’habitants : 68,6 millions (au 1er janvier 2025).
- Nombre de régions : 18 (13 en métropole et 5 en outre-mer).
- Nombre de départements : 101 (96 en métropole et 5 en outre-mer).
- Nombre de communes : environ 35 000 (France métropolitaine et DROM, au 1er janvier 2025).

---

- Outre mer 5 departments (97 & 98)

- France montropolean 96 departments (01 to 96)
- Ile de france 8 departments (75, 77,78,91,92,93,94,95)

 Île-de-France comprises eight departments: 
| Department Code | Department Name   |
| --------------- | ----------------- |
| 75              | Paris             |
| 77              | Seine-et-Marne    |
| 78              | Yvelines          |
| 91              | Essonne           |
| 92              | Hauts-de-Seine    |
| 93              | Seine-Saint-Denis |
| 94              | Val-de-Marne      |
| 95              | Val-d'Oise        |


<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### 🔲 DATASET

| Field Name                     | Python data type             | Definition             |
|--------------------------------|------------------------------|------------------------|
| **Date mutation**              | `object` to `date`       | Date of the property transaction (sale date). |
| **Nature mutation**            | `object` to `str`            | Type of transaction (e.g., Vente, Echange, Donation). |
| **Valeur fonciere**            | `float64`          | Transaction price of the property in euros (€). |
| **Commune**                    | `object` to `str`            | Official name of the commune (city/town). |
| **Type local**                 | `object` to `str`            | Property type: Appartement, Dépendance, Maison, or Local industriel, commercial ou assimilé. |
| **Identifiant local**          | `float64` to `str`            | Unique identifier of the property unit (can be used to track or deduplicate records). |
| **Surface reelle bati**        | `float64`          | Built surface area of the property in square meters (m²). |
| **Nombre pieces principales**  | `float64` to `int`            | Number of main rooms (excluding kitchens, bathrooms, etc.). |
| **Surface terrain**            | `float64`          | Land surface area in square meters (m²), mainly relevant for houses and land. |
| **Nombre de lots**             | `int64`            | Number of lots involved in the transaction (useful for multi-unit sales). |
| **Voie**                       | `object` to `str`            | Street name of the property. |
| **No voie**                    | `float64` to `str`            | Street number of the property. |
| **B/T/Q**                      | `object` to `str`            | Additional address detail: Bâtiment (building), Type, or Quartier (section/block). |
| **Code postal**                | `float64` to `str`            | French postal code identifying the location. |
| **Departement**                | `str`            | First two digits of the postal code representing the department (administrative region). |
| **Prix_m2**                | `float64`            | First two digits of the postal code representing the department (administrative region). |


<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### ➕ COMBINE - 6 TXT FILES INTO 1 CSV
Dropped unnecessary columns

In [ ]:
import pandas as pd

# List of DVF files
dvf_files = [
    "../data/raw/ValeursFoncieres-2020-S2.txt",
    "../data/raw/ValeursFoncieres-2021.txt",
    "../data/raw/ValeursFoncieres-2022.txt",
    "../data/raw/ValeursFoncieres-2023.txt",
    "../data/raw/ValeursFoncieres-2024.txt",
    "../data/raw/ValeursFoncieres-2025-S1.txt"
]

# Columns to keep
usecols = [
    "Date mutation",
    "Nature mutation",
    "Valeur fonciere",
    "Commune",
    "Type local",
    "Surface reelle bati",
    "Nombre pieces principales",
    "Nombre de lots",
    "Surface terrain", 
    "No voie",    # street number
    "B/T/Q",      # building / type / quartier
    "Voie",       # street name
    "Code postal" # postal code
]

# Output CSV
output_file = "../data/processed/RAW_ValeursFoncieres_2020-S2_2025-S1.csv"

# Fix mixed types
dtype_dict = {
    "Code postal": str,
    "Voie": str,
    "No voie": str,
    "B/T/Q": str,
    "Type local": str,
    "Department": str,
    "surface_type": str

}

with open(output_file, "w", newline="", encoding="utf-8") as f_out:
    header_written = False

    for file in dvf_files:
        print(f"Processing: {file}")

        for chunk in pd.read_csv(
            file,
            sep="|",
            usecols=usecols,
            chunksize=50_000,
            encoding="latin-1",
            decimal=",",
            low_memory=False,
            dtype=dtype_dict
        ):
            # -------------------------
            # FIX ENCODING
            # -------------------------
            chunk["Type local"] = (
                chunk["Type local"]
                .astype(str)
                .str.encode("latin-1", errors="ignore")
                .str.decode("utf-8", errors="ignore")
                .str.strip()
            )

            # -------------------------
            # CONVERT NUMERIC
            # -------------------------
            chunk["Valeur fonciere"] = pd.to_numeric(chunk["Valeur fonciere"], errors="coerce")
            chunk["Surface reelle bati"] = pd.to_numeric(chunk["Surface reelle bati"], errors="coerce")
            chunk["Surface terrain"] = pd.to_numeric(chunk["Surface terrain"], errors="coerce")

            # Drop rows with missing price
            chunk = chunk.dropna(subset=["Valeur fonciere"])

            # -------------------------
            # PRICE PER M² (SMART FALLBACK)
            # -------------------------
            chunk["surface_used"] = chunk["Surface reelle bati"]

            # If built surface is missing or zero → use terrain
            mask = chunk["surface_used"].isna() | (chunk["surface_used"] == 0)
            chunk.loc[mask, "surface_used"] = chunk.loc[mask, "Surface terrain"]

            # Compute price per m²
            chunk["Prix_m2"] = chunk["Valeur fonciere"] / chunk["surface_used"]

            # Clean invalid values
            chunk.loc[
                chunk["surface_used"].isna() | (chunk["surface_used"] == 0),
                "Prix_m2"
            ] = None

            # Optional: track which surface was used
            chunk["surface_type"] = "bati"
            chunk.loc[mask, "surface_type"] = "terrain"

            # -------------------------
            # EXTRACT DEPARTEMENT
            # -------------------------
            chunk["departement"] = chunk["Code postal"].astype(str).str[:2]

            # -------------------------
            # WRITE TO CSV
            # -------------------------
            chunk.to_csv(
                f_out,
                index=False,
                header=not header_written
            )

            header_written = True
            del chunk  # free memory

print(f"✅ File saved at {output_file}")

Processing: ../data/raw/ValeursFoncieres-2020-S2.txt
Processing: ../data/raw/ValeursFoncieres-2021.txt
Processing: ../data/raw/ValeursFoncieres-2022.txt
Processing: ../data/raw/ValeursFoncieres-2023.txt
Processing: ../data/raw/ValeursFoncieres-2024.txt
Processing: ../data/raw/ValeursFoncieres-2025-S1.txt
✅ File saved at ../data/processed/RAW_ValeursFoncieres_2020-S2_2025-S1.csv


<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### 🔲 DISPLAY - HEAD

In [2]:
import pandas as pd
df = pd.read_csv("../data/processed/RAW_ValeursFoncieres_2020-S2_2025-S1.csv")

# Display df DataFrame
print("DATA EXPLORATION:")
display(df.head())
print("Dataset Shape:", df.shape[0], "rows and", df.shape[1], "columns\n")

C:\Users\sboub\AppData\Local\Temp\ipykernel_8328\1009436813.py:2: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/RAW_ValeursFoncieres_2020-S2_2025-S1.csv")


DATA EXPLORATION:


,Date mutation,Nature mutation,Valeur fonciere,No voie,B/T/Q,Voie,Code postal,Commune,Nombre de lots,Type local,Surface reelle bati,Nombre pieces principales,Surface terrain,surface_used,Prix_m2,surface_type,departement
0,01/07/2020,Vente,31234.16,NaN,NaN,SAINT JULIEN,1560.0,SAINT-JULIEN-SUR-REYSSOUZE,0,NaN,NaN,NaN,1192.0,1192.0,26.203154,terrain,15
1,01/07/2020,Vente,278000.00,NaN,NaN,A LA PEROUSE,1250.0,CORVEISSIAT,0,NaN,NaN,NaN,10092.0,10092.0,27.546572,terrain,12
2,01/07/2020,Vente,278000.00,NaN,NaN,A LA PEROUSE,1250.0,CORVEISSIAT,0,NaN,NaN,NaN,4570.0,4570.0,60.831510,terrain,12
3,01/07/2020,Vente,278000.00,NaN,NaN,AUX COMMUNS,1250.0,CORVEISSIAT,0,NaN,NaN,NaN,5750.0,5750.0,48.347826,terrain,12
4,01/07/2020,Vente,278000.00,NaN,NaN,EN COMBARNAUD,1250.0,SIMANDRE-SUR-SURAN,0,NaN,NaN,NaN,648170.0,648170.0,0.428900,terrain,12


Dataset Shape: 19908349 rows and 17 columns



<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### 🔲 DISPLAY - UNIQUE VALUES

In [3]:
# Display data types & missing values of each column in df
print("Data Types & Missing Values of Each Column:")
display(df.info())

# Loop through each column and print unique values
for col in df.columns:
    unique_vals = df[col].unique()
    print(f"Column: {col}")
    print(f"Unique values ({len(unique_vals)}): {unique_vals}\n")

print(f"\nData types check:")
for col in df.columns:
    dtype = df[col].dtype
    unique_vals = df[col].nunique()
    print(f"  ➡️{col:<25} {str(dtype):<10} [{unique_vals}] unique values")

Data Types & Missing Values of Each Column:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19908349 entries, 0 to 19908348
Data columns (total 17 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   Date mutation              object 
 1   Nature mutation            object 
 2   Valeur fonciere            float64
 3   No voie                    float64
 4   B/T/Q                      object 
 5   Voie                       object 
 6   Code postal                float64
 7   Commune                    object 
 8   Nombre de lots             int64  
 9   Type local                 object 
 10  Surface reelle bati        float64
 11  Nombre pieces principales  float64
 12  Surface terrain            float64
 13  surface_used               float64
 14  Prix_m2                    float64
 15  surface_type               object 
 16  departement                object 
dtypes: float64(8), int64(1), object(8)
memory usage: 2.5+ GB


None

Column: Date mutation
Unique values (1804): ['01/07/2020' '04/07/2020' '02/07/2020' ... '01/05/2025' '18/05/2025'
 '05/01/2025']

Column: Nature mutation
Unique values (6): ['Vente' "Vente en l'Ã©tat futur d'achÃ¨vement"
 'Vente terrain Ã\xa0 bÃ¢tir' 'Echange' 'Adjudication' 'Expropriation']

Column: Valeur fonciere
Unique values (466667): [3.123416e+04 2.780000e+05 3.985000e+03 ... 7.088090e+05 5.503861e+05
 2.441758e+07]

Column: No voie
Unique values (9121): [  nan  347. 1084. ... 8562. 8301. 8423.]

Column: B/T/Q
Unique values (41): [nan 'B' 'C' 'A' 'F' 'L' 'T' 'X' 'O' 'Z' 'D' 'G' 'Q' 'Y' 'I' 'E' 'N' 'H'
 '1' 'W' 'V' 'U' 'S' 'R' 'P' 'M' 'K' 'J' '0' '5' '9' '8' 'b' '7' '2' '4'
 '3' '6' '-' '.' '*']

Column: Voie
Unique values (1125220): ['SAINT JULIEN' 'A LA PEROUSE' 'AUX COMMUNS' ... 'SAINT PIERRE AMELOT'
 'SAINT HONORE D EYLAU' 'DES FOSSES SAINT MARCEL']

Column: Code postal
Unique values (5872): [ 1560.  1250.  1310. ... 97380. 97314. 97330.]

Column: Commune
Unique values (31569

data exploration notebook creates:
RAW_DVF (only combined 6 txt files to 1 csv with columns selected and adding surface_used, Prix_m2,surface_type, departement)

data cleaning notebook creates:
STANDARD_DVF

CLEAN_DVF ()

--------------------------------
DVF NOTES FOR CLEANING:


- standardize column names
| Original Column           | Improved Name     | Notes / Description                                          |
| ------------------------- | ----------------- | ------------------------------------------------------------ |
| Date mutation             | transaction_date  | Date when the property was sold                              |
| Nature mutation           | transaction_type  | Nature of transaction (sale, donation, etc.)                 |
| Valeur fonciere           | sale_price_eur    | Transaction price in euros                                   |
| No voie                   | street_number     | Street number of the property                                |
| B/T/Q                     | building_code     | Bâtiment / Type / Quartier code for identification           |
| Voie                      | street_name       | Full street name                                             |
| Code postal               | postal_code       | Postal code (categorical, not numeric)                       |
| Commune                   | city              | Commune / city name                                          |
| Nombre de lots            | number_of_lots    | Number of lots in the property                               |
| Type local                | property_type     | Property type (Apartment, House, etc.)                       |
| Surface reelle bati       | building_area_m2  | Built area in square meters                                  |
| Nombre pieces principales | main_rooms        | Number of main rooms (living / bedrooms)                     |
| Surface terrain           | land_area_m2      | Land area in square meters                                   |
| surface_used              | effective_area_m2 | Surface used to calculate price/m² (building or land)        |
| Prix_m2                   | price_per_m2      | Price per m² in euros                                        |
| surface_type              | area_type         | Indicates if `effective_area_m2` comes from building or land |
| departement               | department_code   | 2-digit INSEE department code                                |



- convert types
    - Column: Code postal to object
    - Column: Nombre pieces principales to int
    - Column: Date mutation to date? is it good for ML?
    - Column: No Voie to int 
    like so:
    ➡️Date mutation             object     date
    ➡️Nature mutation           object     
    ➡️Valeur fonciere           float64    
    ➡️No voie                   float64    int64
    ➡️B/T/Q                     object     
    ➡️Voie                      object     
    ➡️Code postal               float64    object
    ➡️Commune                   object     
    ➡️Nombre de lots            int64      
    ➡️Type local                object     
    ➡️Surface reelle bati       float64    
    ➡️Nombre pieces principales float64    int64
    ➡️Surface terrain           float64    
    ➡️surface_used              float64    
    ➡️Prix_m2                   float64    
    ➡️surface_type              object     
    ➡️departement               object



- deal with invalid values

    - Column: Nature mutation 
    Unique values (6): ['Vente' "Vente en l'Ã©tat futur d'achÃ¨vement" 'Vente terrain Ã\xa0 bÃ¢tir' 'Echange' 'Adjudication' 'Expropriation'] 

    - Column: surface_type
    Unique values (2) : 'bati' to 'batiment'

    - Column: B/T/Q
    Unique values (41): [nan 'B' 'C' 'A' 'F' 'L' 'T' 'X' 'O' 'Z' 'D' 'G' 'Q' 'Y' 'I' 'E' 'N' 'H'
     '1' 'W' 'V' 'U' 'S' 'R' 'P' 'M' 'K' 'J' '0' '5' '9' '8' 'b' '7' '2' '4' '3' '6' '-' '.' '*']
    check for what the hell is up with the symbols '-' '.' '*'



- deal with duplicates:
    - Column: Commune
    check for any of them having the same commune but differently written:
    Unique values (31569): ['SAINT-JULIEN-SUR-REYSSOUZE' 'CORVEISSIAT' 'SIMANDRE-SUR-SURAN' ...
    'BEAUMONT SUR OISE' 'FRETTE SUR SEINE (LA)' 'CAMOPI']    



- deal with nulls:
    
    - Column: Type local
    drop nan 
    
    - Column: Valeur fonciere
    check for any NaN, if there are then drop them
    
    - Column: Surface reelle bati 
    Fill NaN with "0"
    
    - Column: Type local 
    Fill NaNs with "Terrain" or "Unknown"

    - Column: Nombre pieces principales
    fill NaN with 0 for terrains or ignore if building type is non-built




SIDE NOTES!
+ I probably will need a table from large to small, Country -- Region -- Department - ile de france - paris - city - street neighbourhood
+ Keep only rows with Valeur fonciere > 0
+ Nombre pieces principales is NaN in the examples. Only relevant for houses/apartments.
Could fill NaN with 0 for terrains or ignore if building type is non-built.


---
## **💲 API RENTAL PRICES & YIELD**

**`INTERIM_API_RENTAL.CSV`**

What it contains:
- Median rent (€ / month) 
- Rent per m² 
- Vacancy rate (%) 
- Rent evolution over time 
- Data by arrondissement / commune

<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### 🔲 DATASET

| Field Name             | Python Data Type       | Concise Definition                                  |
|------------------------|------------------------|-----------------------------------------------------|

In [ ]:
# api for rental prices

---
## **🚇 CSV LIGNES DE TRANSPORT EN COMMUN**

TRANSPORT dataset is used for:
- a
- b
- c


What it contains:
- Metro, RER, tram, bus lines 
- Stops/stations coordinates 
- Line types & frequency (if available)

<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### 🔲 DATASET

| Field Name             | Python Data Type       | Concise Definition                                  |
|------------------------|------------------------|-----------------------------------------------------|

In [ ]:
# LIGNES DE TRANSPORT EN COMMUN
df_lignes_transport = pd.read_csv("../data/raw/traces-des-lignes-de-transport-en-commun-idfm.csv", sep=";")
print("Lignes de transport en commun:")
display(df_lignes_transport.head())

---
## **🌍 API MAPPING PARIS & BEAULIEU PARISIENNE**

What it contains:
- Latitude & longitude boundaries 
- Arrondissements polygons 
- Commune mapping 
- INSEE codes

<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### 🔲 DATASET

| Field Name             | Python Data Type       | Concise Definition                                  |
|------------------------|------------------------|-----------------------------------------------------|

In [ ]:
# api for paris / outside paris
# arrondissements boundaries
# communes ile de france boundaries

---
## **🌍 API MAP AMENITIES**

In [ ]:
# api for ameneties